# Procedimiento

### Lectura del EEG crudo
Parámetros:
 - archivo_r2a: ruta al archivo de ondas crudas.
 - fs=128: frecuencia de muestreo del BIS. Hay 128 muestras por segundo.
 - escala_uv=0.0511: factor de conversión para transformar los valores enteros del archivo binario en microvoltios.
 
Devuelve:
 - df_eeg: DataFrame con columnas: tiempo_s, canal_1_raw, canal_2_raw, canal_1_uV, canal_2_uV.
           señal cruda de los dos canales en microvoltios


### Reconstrucción espectral del canal
Parámetros:
 - df_eeg: contiene el EEG crudo leído antes.
 - "canal_1_uV": se usa la señal del canal 1 en microvoltios.
 - fs=128: frecuencia de muestreo.
 - ventana_seg=2: cada espectro se calcula con ventanas de 2 segundos.
 - paso_seg=1: la ventana avanza cada segundo.
 - fmin=0.5: frecuencia mínima representada.
 - fmax=30.0: frecuencia máxima representada.
 - paso_freq=0.5: resolución frecuencial de 0.5 Hz.
 - modo="densidad": se obtiene densidad espectral de potencia.
 - tiempo_referencia="centro": cada ventana se etiqueta por su centro temporal.
 
Devuelve:
 - df_dsa_canal1: matriz tiempo-frecuencia del canal 1.
 - frecuencias_c1: array con las frecuencias usadas, desde 0.5 hasta 30 Hz.
 
 
### Selección de columnas de frecuencia (cols_freq)
El DataFrame df_dsa_canal1 tiene una columna temporal llamada tiempo_s y luego muchas columnas de frecuencia:
    tiempo_s | 0.5 | 1.0 | 1.5 | ... | 30.0
    
Crear una lista con todas las columnas excepto tiempo_s


### Crear estructura para la DSA media (df_dsa_media)
 - Se hace una copia del df_dsa_canal1 para la conservación de la estructura: tiempo y frecuencias
 - Se usa de base para sobreescribir con la media de los dos canales
 

### Media de potencia de los dos canales (pot_media)
 - Extraer las columnas de frecuencia de cada canal:
     - df_dsa_canal1[cols_freq]
     - df_dsa_canal2[cols_freq]
 - Convertir a array numérico: 
     - .to_numpy(dtype=float)
 - Sumar ambos canales y dividir entre dos
Devuelve: pot_media = matriz de numpy tiempo x frecuencia


### Convertir frecuencias a número (frecuencias_float)
 - El cabecero estaba escrito en formato texto y se pasa a numérico
Devuelve: array numérico con las frecuencias


### Cálculo de SEF y MEF propios
Parámetros:
 - potencia=pot_media: matriz de potencias en función del tiempo en escala lineal.
 - frecuencias=frecuencias_float: frecuencias correspondientes a cada columna.
 - percentil_sef=0.95: SEF es la frecuencia por debajo de la cual se acumula el 95 % de la potencia.
 - percentil_mef=0.50: MEF es la frecuencia por debajo de la cual se acumula el 50 % de la potencia.

Devuelve:
 - sef_eeg: array con el valor del parámetro por segundo 
 - mef_eeg: array con el valor valor del parámetro por segundo
 
Como la reconstrucción con ventanas de 2 segundos requiere que se añada una fila inicial NaN, como queremos que el SEF y el MEF tenga la misma longitud que el SEF, también se le añade un NaN al principio.
 - np.r_[np.nan, sef_eeg]: concatena un NaN inicial con el vector sef_eeg.
 - pd.Series(...): lo convierte en una serie de pandas.
 - name="SEF08": le da nombre compatible con tu función de plot.

Devuelve:
 - mef_eeg_plot: serie con el mef propio preparado para meter al plot
 - sef_eeg_plot: serie con el sef propio preparado para meter al plot
 
 
### Conversión de potencia a dB (df_dsa_media[cols_freq])
Se transforma la potencia media lineal a decibelios:
 - pot_media: matriz de potencia/densidad en escala lineal.
 - 1e-12: valor muy pequeño para evitar log10(0).
 - ref_uv_rms ** 2: potencia.
 - 10 * np.log10(...): conversión a decibelios.
 
El resultado se guarda en el dF con la media de potencias y sobreescribiendo el contenido de las columnas.

### Preparar el .spa temporalmente (fun_dsa.obtener_hora_inicio_desde_spa)
Parámetros:
 - df_spa_unilat: dF con las variables procesadas

Devuelve:
 - df_spa_unilat: el DataFrame .spa preparado, con la columna Time en formato datetime y redondeada al segundo.
 - hora_inicio: hora inicial.

Se limpia el .spa para quedarnos con las filas que sí pueden alinearse temporalmente. Después, la DSA mantiene su propio eje temporal continuo con tiempo_eeg en posteriores pasos.
 

### Adaptar DSA reconstruida al tiempo real (fun_dsa.adaptar_dsa_reconstruida_para_plot)
Parámetros:
 - df_dsa=df_dsa_media: dF con la DSA reconstruida en dB.
 - frecuencias=frecuencias_c1: frecuencias de la matriz.********************** POR QUÉ NO SE USA FRECUENCIAS_FLOAT
 - hora_inicio=hora_inicio: hora inicial.
 - insertar_fila_inicial_nan=True: añade una fila inicial con NaN para compensar que la primera ventana centrada cae en el segundo 1. ************************ EXPLICACIÓN EN EL CUADERNO

Devuelve:
 - tiempo_eeg: serie temporal real, con fechas y horas. ************************** SERIE DE PANDAS?
 - dsa_eeg: dF matriz DSA en dB, ya sin columna tiempo_s. ********************************** DF?

 
### Alinear .spa con la DSA reconstruida (fun_dsa.alinear_spa_con_tiempo)
Alinear las variables del .spa con los tiempos de la DSA reconstruida.

Se crea una tabla con todos los tiempos de la DSA. Si en el .spa falta información para algún segundo, ese segundo no desaparece: queda en la DSA, pero con valores del .spa como NaN. 
 - Por eso el merge se hace con how="left": conserva todos los tiempos de la DSA aunque el .spa no tenga datos para alguno de ellos.

Parámetros:
 - tiempo=tiempo_eeg: serie temporal. Fecha y hora de cada registro. Un registro cada segundo
 - df_spa=df_spa_unilat: el DataFrame .spa preparado, con la columna Time en formato datetime y redondeada al segundo.

Devuelve:
 - df_merge_plot : DataFrame con una fila por cada instante de tiempo_eeg y añade las variables del .spa que coinciden en ese segundo.
 
Usarán variables del SPA como:  SQI10, TOTPOW08, SEF08 en la misma línea temporal que la DSA.


### Sustituir SEF y MEF por los propios
 - mef_eeg_plot.values: recoge los valores mef de la serie
 - sef_eeg_plot.values: recoge los valores sef de la serie
 
Devuelve:
 - df_merge_plot["MEDFRQ08"]: como el spa tenía valores SEF y MEF procesados, los cambia por los calculados
 
 
### Calcular máscara total (fun_dsa.preparar_dsa_con_mask)
La máscara marca como no válidos los segundos donde se cumplen criterios

Parámetros:
 - tiempo=tiempo_eeg: serie temporal. Fecha y hora de cada registro. Un registro cada segundo
 - dsa=dsa_eeg: matriz DSA en dB, ya sin columna tiempo_s.
 - df_merge=df_merge_plot: df_merge_plot : DataFrame con una fila por cada instante de tiempo_eeg y añade las variables del .spa que coinciden en ese segundo.
 - umbral_sqi=15: criterio de invalidez de fila
 - umbral_ceros=0.9: criterio de invalidez de fila

Devuelve:
 - dsa preparada para plot: no hace falta así que se suele poner un _
 - mask_total: serie booleana (False=fila válida, True=fila no válida)
 
 
### Crear máscara común
Utiliza:
 - mask_total.copy(): copia de la máscara total para usarla como máscara común en todas las matrices.

Devuelve:
 - mask_comun: evitamos aplicar máscaras distintas en diferentes puntos del flujo.
 
 
### Aplicar máscara común a la DSA original f_a

### Preparar escala de color para la DSA reconstruida (fun_dsa.preparar_escala_color_dsa)
Parámetros:
 - dsa_eeg_directa_plot:
 - gamma: parámetro que modifica la distribución de colores para mejorar el parecido con la DSA del BIS. No cambia los datos, solo la representación.

Devuelve:
 - matriz_eeg: matriz NumPy para imshow.
 - vmin_eeg: valor mínimo de la escala de color.
 - vmax_eeg: valor máximo de la escala de color.
 - norm_eeg: normalización de color.
 - cmap_eeg: colormap tipo BIS.

# Reconstrucción

In [1]:
"""
Tipo de salida de Welch
 - Welch no te devuelve "la frecuencia". Welch te devuelve cuánta energía/potencia de la señal hay asociada a cada frecuencia.
 - La potencia se puede expresar de dos maneras: 
     - density  → potencia por Hz
     - spectrum → potencia por bin de frecuencia

scaling = "density" if modo == "densidad" else "spectrum" // cuánta potencia hay por cada Hz vs cuánta potencia hay en este intervalo concreto

 - Si modo es: "densidad" o "db_densidad"
     - scaling="density"
     - devuelve densidad espe3ctral de potencia, normalmente interpretable como potencia por Hz. 
     
     - Representa cuánta potencia hay por unidad de frecuencia, por cada Hz de ancho de banda. µV²/Hz
     - Útil para una estimación espectral comparable entre resoluciones frecuenciales.
     
 - Si modo es: "potencia", "db" o "amplitud"
     - scaling="spectrum"
     - devuelve potencia espectral integrada por frecuencia. 
     
     - Representa potencia integrada por bin de frecuencia. Representa la potencia contenida en ese intervalo frecuencial concreto. µV²
     - Un bin es cada “cajita” de frecuencia que te da la FFT/Welch. Como paso_freq = 0.5, cada bin representa un intervalo de 0.5 Hz.
     
En este trabajo se usa modo="densidad", por lo que Welch devuelve una densidad espectral lineal.
Posteriormente, tras promediar los canales y calcular métricas sobre la potencia, se transforma la matriz a dB para visualizar la DSA.

"""

'\nTipo de salida de Welch\n - Welch no te devuelve "la frecuencia". Welch te devuelve cuánta energía/potencia de la señal hay asociada a cada frecuencia.\n - La potencia se puede expresar de dos maneras: \n     - density  → potencia por Hz\n     - spectrum → potencia por bin de frecuencia\n\nscaling = "density" if modo == "densidad" else "spectrum" // cuánta potencia hay por cada Hz vs cuánta potencia hay en este intervalo concreto\n\n - Si modo es: "densidad" o "db_densidad"\n     - scaling="density"\n     - devuelve densidad espe3ctral de potencia, normalmente interpretable como potencia por Hz. \n     \n     - Representa cuánta potencia hay por unidad de frecuencia, por cada Hz de ancho de banda. µV²/Hz\n     - Útil para una estimación espectral comparable entre resoluciones frecuenciales.\n     \n - Si modo es: "potencia", "db" o "amplitud"\n     - scaling="spectrum"\n     - devuelve potencia espectral integrada por frecuencia. \n     \n     - Representa potencia integrada por bin

## Ventana de Hann
Aplica una ventana de Hann al segmento antes de calcular el espectro.

Esto reduce discontinuidades en los bordes de la ventana y ayuda a disminuir fugas espectrales.

### nperseg=nperseg 
Indica el tamaño del segmento usado por Welch. Aquí coincide con la ventana completa: 256 muestras

### noverlap=0
Indica que dentro de esa llamada a Welch no hay solapamiento entre subsegmentos.

Como cada segmento ya mide 2 segundos y nperseg también mide 2 segundos, Welch calcula un espectro por segmento completo.

El solapamiento temporal real lo introduces tú fuera de Welch, porque el bucle avanza 1 segundo con ventanas de 2 segundos.



## Eliminar la componente continua (detrend="constant")
Para cada ventana de 2 segundos, antes de calcular el espectro, se resta el valor medio de esa ventana. Esto ayuda a quitar la componente DC.


Para reconstruir una DSA, tú quieres saber cuánta potencia hay en frecuencias como: 0.5 Hz, 1 Hz, 2 Hz, 4 Hz, 8 Hz, 13 Hz...

Pero si la señal tiene un valor medio muy alto, aparece mucha energía artificial en frecuencias muy bajas, especialmente cerca de 0 Hz. Puede distorsionar el espectro.

Explicación:
Si tu EEG original está flotando sobre un voltaje base de 50 milivoltios debido a la conductividad de la piel del paciente o a la estática del cable, cuando el ordenador ponga la plantilla de 0 Hz encima para buscar un match, la multiplicación dará un número gigantesco.


Contaminación de las bandas delta:


# FT / DFT / FFT
**Serie de Fourier:** Es una técnica o idea que dice que cualquier función compleja, siempre que sea periódica, se puede "descomponer" o representar como una suma de ondas simples (senos y cosenos). Cada una de estas ondas simples tiene su propia amplitud, frecuencia y fase.

FT, DFT y FFT no son tres cosas totalmente distintas.
Son tres niveles de la misma idea:
 - FT = transformada matemática para señales continuas.
 - DFT = versión para señales discretas y finitas, como datos de EEG muestreados.
 - FFT = algoritmo rápido para calcular la DFT, no una transformada diferente.


## FT
Convierte una función del tiempo a su representación en el dominio de la frecuencia. En vez de describir la señal como “qué valor tiene en cada instante”, la describe como “qué frecuencias contiene y con qué intensidad”.

Permite descomponer una señal en sus componenetes sinusoidales. Para una función continua, la transformada se define a través de:

$$
X(f) = \int_{-\infty}^{\infty} x(t) e^{-i 2 \pi f t} \, dt
$$

Donde:
 - t: tiempo. Variable del dominio original. El instante temporal en el que se mide la señal (1/128 segundos)
 - f: frecuencia. Variable del nuevo dominio. Ciclos por segundo (Hz) a los que oscila cada componenete de la señal
 
 - x(t):señal en el tiempo. Señal de entrada. Valor de la amplitud de la señal en cada momento del tiempo.
 - X(f): espectro de frecuencia. Resultado de la transformada. Cantidad de cada frecuencia "f" que tiene la señal original. Nº complejo que contiene dos tipos de información: 
   - amplitud: |X(f)|
   - fase: ∠ X(f) (ángulo)
 
 - $$\(\int_{-\infty}^{\infty} ... dt\)$$: integral infinita. Significa que se analiza la señal a lo largo de todo el tiempo. Suma y promedia las coincidencias de la señal original con la frecuencia (f) buscada
 
 - $$\(e^{-i2\pi ft}\)$$: Basado en la identidad de Euler. Actúa como una sonda de frecuencia que analiza la señal original.
     - concepto de giro (Identidad de Euler): equivale a un punto que se mueve continuamente describiendo círculos de radio 1 en el plano complejo a una velocidad y ritmo determinados por la frecuencia $f$.
     
     - unidad Imaginaria (\(i\)): la base del plano complejo $(\(\sqrt{-1}\))$. Permite dividir la información de la señal en dos dimensiones perpendiculares: la parte real/Amplitud mediante el coseno y la parte imaginaria/Fase mediante el seno.
     
     - factor de Escala (\(2 \pi\)): al multiplicar la frecuencia lineal \(f\) (Hz o ciclos/segundo) por \(2\pi\), se transforma en frecuencia angular (\(\omega = 2\pi f\)), permitiendo que el cálculo del movimiento circular sea matemáticamente exacto.
     
     - signo (negativo) (\(-i\)): establece la dirección de giro y operación de análisis: descomposición de la señal temporal para convertir al dominio de la frecuencia (negativo) o reconstrucción (positivo).
     
La FT ideal trabaja con señales continuas e infinitas en el tiempo. Pero en un ordenador no tenemos una función continua ni infinita.

En un caso como EEG, tenemos una lista finita de muestras. Por ejemplo, si la frecuencia de muestreo es de 128 Hz, significa que se toman 128 valores por segundo.

Por eso se usa la DFT, que es la versión discreta y finita de la Transformada de Fourier.


## DFT
La DFT transforma una secuencia finita de muestras en una secuencia finita de componentes de frecuencia.
Descompone la señal del EEG en todas las frecuencias que la componen. Convierte esta señal en una especie de mapa que indica cuánto hay (amplitud) de esa frecuencia y dónde está (fase) dicha frecuencia.

Al tener una señal discreta:
x_0, x_1, x_2, ..., x_N-1

su DFT se define como:

$$
X_k = \sum_{n=0}^{N-1} x_n e^{-i 2\pi k n / N}
$$

Donde:
 - \(X[k]\): cantidad de la frecuencia k presente en la señal.
 - N: total de muestras
 - $n$: índice de cada muestra temporal.
 - $x_n$:  el valor de tu EEG en el punto número $n$. Señal original en el dominio del tiempo.
 - $k$: índice de frecuencia. 
 

La frecuencia real asociada a cada $k$:      
 - f_k = k x F_s / N

Donde:
 - Fs: frecuencia de muestreo.
 - N: número de muestras.
 - k: índice de frecuencia.
 
Si el EEG se muestrea a Fs = 128 Hz y se analiza una ventana de N = 256 muestras, la resolución frecuencial será:
 - Af = Fs / N = 128 / 256 = 0.5 Hz
 
Por tanto la DFT nos dará valores para ciertas frecuencias concretas: 0 Hz, 0.5 Hz, 1 Hz, 1.5 Hz, ...
hasta el límite de Nyquist (frecuencia máxima que se puede capturar con precisión al digitalizar una señal analógica):
 - f_Nyquist = Fs / 2 = 128 / 2 = 64 Hz

### Nyquist
Por tanto, aunque la DFT de 256 muestras devuelve 256 valores X_k, en una señal real como el EEG normalmente se interpreta solo la primera mitad del espectro, desde 0 Hz hasta 64 Hz.
Como la resolución es de 0.5 Hz, las frecuencias interpretables serán: 0 Hz, 0.5 Hz, 1 Hz, 1.5 Hz, ..., 63.5 Hz, 64 Hz.

La segunda mitad del resultado de la DFT no representa nuevas frecuencias positivas, sino frecuencias negativas. Para una señal real, esas frecuencias negativas contienen la misma información que las positivas, pero reflejada. Por eso, en análisis de EEG, normalmente se trabaja solo con el rango de 0 Hz a Fs/2.

#### Razón matemática
En la DFT, los índices de frecuencia son periódicos. Por eso, en una FFT de N muestras, los índices mayores que N/2 se interpretan como frecuencias negativas. Por ejemplo, con Fs=128 Hz y N=128, 
 - X_128 corresponde a −64 Hz (Frec. Nyquist) y X_255 corresponde a −0.5 Hz. 
En señales reales, estas frecuencias negativas son el reflejo conjugado de las positivas:
 - X[N - k] = X[K]*
Por lo que normalmente se analiza solo el rango de 0 Hz a Fs/2.


## Transformada Rápida de Fourier (FFT)
